read from translated top sun list that look like this: wildtype_aa1	pos1	mutate_aa1	wildtype_aa2	pos2	mutate_aa2	DDE	untranslated_wildtype_aa1	untranslated_mutate_aa1	untranslated_wildtype_aa2	untranslated_mutate_aa2
B	30	D	B	88	C	4.6108785	D	N	N	D
C	32	D	A	47	B	3.4506056	V	I	I	V
B	48	A	C	54	A	3.1376057	G	V	I	A
B	30	D	C	45	D	3.081643	D	N	K	Q,  get the list of double mutation string in format of "D30N-N88D"

In [3]:
import pandas as pd
from io import StringIO

# Read the data into a DataFrame from a TSV file
df = pd.read_csv("data/translated_top_syn_list.tsv", sep="\t", comment = '#')

# Generate the double mutation strings
double_mutations = df.apply(
    lambda row: f"{row['untranslated_wildtype_aa1']}{row['pos1']}{row['untranslated_mutate_aa1']}_"
                f"{row['untranslated_wildtype_aa2']}{row['pos2']}{row['untranslated_mutate_aa2']}", axis=1)

# Convert to a list
double_mutation_list = double_mutations.tolist()
double_mutation_tuples = list(zip(double_mutation_list, df['DDE']))
print(double_mutation_tuples)

[('K101E_G190S', 3.3036745), ('K101E_G190A', 2.642376), ('K103N_P225H', 2.635779), ('L100I_K103N', 2.58845), ('K101P_K103S', 2.3547902), ('Y181C_H221Y', 1.7625897), ('K103S_G190A', 1.7216139), ('K103S_P225H', 1.6469941), ('L100I_K103R', 1.4893179), ('V108I_H221Y', 1.4863837), ('K103S_D192N', 1.4760622), ('L100I_K103S', 1.3615851), ('K101E_E138A', 1.2651696), ('Y181C_G190A', 1.1883571), ('K103S_D177N', 1.1777711), ('V108I_V189I', 1.1743443), ('K101E_E138K', 1.1518887), ('E138A_G190E', 1.0947237), ('K101P_D192N', 1.0646284), ('V108I_L109V', 1.0552669)]


read the mutation from double_mutation tuple list and output like this in a tsv form:
                    
Mutation Pairs	ΔΔE	Energy Type	ΔE(M1,M2)	ΔE(M1)	ΔE(M2)
D30N-N88D		Wildtype			
D30N-N88D		Rescue example			
D30N-N88D		Compensate example			
D30N-N88D		Antagonistic example			
D30N-N88D		flip example			                    
V32I-I47V		

for each pair, Delta delta E is in the tuple. and each has a row of energy type fixed to the example above. To access dm12, dm1, dm2,read into corrosponding folder: for wildtype its data/wildtype_out/{mutation name}.tsv, for rescue its data/rescue_out/{mutation name}.tsv and so on, access the first rows value and put in a new summary.tsv file. 

In [4]:
import os

# Define the energy types and their corresponding folders
energy_types = [
    ("Wildtype", "consensus_out"),
    ("Rescue example", "rescue_out"),
    ("Compensate example", "compensate_out"),
    ("Antagonistic example", "antag_out"),
    ("flip example", "flip_out"),
]

# Prepare the output data
output_data = []

for mutation, dde in double_mutation_tuples:
    for energy_type, folder in energy_types:
        # Construct the file path
        file_path = f"data/{folder}/{mutation}.tsv"
        
        # Read the first row value if the file exists
        if os.path.exists(file_path):
            with open(file_path, 'r') as file:
                first_row = file.readline().strip()
                second_row = file.readline().strip()
            # print(f"Read from {file_path}: {first_row}")
        else:
            # print(f"File not found: {file_path}")
            first_row = ""
        
        # Append the row to the output data
        first_row_values = first_row.split("\t")
        second_row_values = second_row.split("\t") if second_row else []
        
        # Safely access indices with default values if they are out of range
        delta_e_m1_m2 = second_row_values[5] if len(second_row_values) > 5 else ""
        delta_e_m1 = second_row_values[3] if len(second_row_values) > 3 else ""
        delta_e_m2 = second_row_values[4] if len(second_row_values) > 4 else ""
        
        output_data.append([mutation, dde, energy_type, delta_e_m1_m2, delta_e_m1, delta_e_m2])
    # print(output_data)

# Create a DataFrame for the output
output_df = pd.DataFrame(output_data, columns=["Mutation Pairs", "ΔΔE", "Energy Type", "ΔE(M1,M2)", "ΔE(M1)", "ΔE(M2)"])

# Save the DataFrame to a TSV file
parent_directory = os.path.dirname(os.path.abspath("__file__"))
parent_directory_name = os.path.basename(parent_directory)
# print(parent_directory_name)
output_df.to_csv(f"data/{parent_directory_name}_energy_summary.tsv", sep="\t", index=False)